# Oil Well Location Selection

Using Bootstrap analysis to choose the most profitable drilling region.

## Project Overview

You work at an oil extraction company and need to decide where to drill a new well.

**Data:** Oil samples from three regions, 10,000 deposits each, with quality measurements and reserve volumes.

**Goal:** Build a machine learning model to identify the region with the highest profit potential, then analyze profit and risks using Bootstrap.

**Selection Process:**
1. Explore deposits in each region and measure features
2. Build a model to predict reserve volumes
3. Select deposits with highest predicted values (within budget constraints)
4. Calculate total profit from selected deposits

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create images directory
os.makedirs('reports/images', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

## Data Loading and Preparation

In [ ]:
# Load data from CSV files
file_paths = ["../../datasets/geo_data_0.csv", "../../datasets/geo_data_1.csv", "../../datasets/geo_data_2.csv"]
data = [pd.read_csv(path) for path in file_paths]

In [ ]:
for i, df in enumerate(data):
    print(f"Region {i}:")
    print(df.info())
    print(df.describe())
    print("-" * 50)

In [ ]:
# Visualize reserve distribution by region
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, (df, ax, color) in enumerate(zip(data, axes, colors)):
    ax.hist(df['product'], bins=50, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(df['product'].mean(), color='black', linestyle='--', linewidth=2, label=f'Mean: {df["product"].mean():.1f}')
    ax.set_title(f'Region {i}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Oil Reserves (thousand barrels)')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Oil Reserve Distribution by Region', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('reports/images/reserve_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

**Data Summary:**
- **Total records:** 100,000 per region
- **Average oil reserves (thousand barrels):**
    - Region 0: **92.50** (std: 44.29)
    - Region 1: **68.82** (std: 45.94)
    - Region 2: **95.00** (std: 44.75)

**Insight:** Region 2 has the highest average reserves, Region 1 the lowest. All regions show high variance.

## Model Training and Evaluation

In [ ]:
# Split data into training and validation sets
train_dfs, valid_dfs = [], []
for df in data:
    train, valid = train_test_split(df, test_size=0.25, random_state=42)
    train_dfs.append(train)
    valid_dfs.append(valid)

In [ ]:
# Function to train model and calculate RMSE
def train_and_evaluate(train_df, valid_df):
    features_train, target_train = train_df.drop(['id', 'product'], axis=1), train_df['product']
    features_valid, target_valid = valid_df.drop(['id', 'product'], axis=1), valid_df['product']

    model = LinearRegression()
    model.fit(features_train, target_train)
    predictions = model.predict(features_valid)

    rmse = mean_squared_error(target_valid, predictions) ** 0.5
    mean_predicted = predictions.mean()
    return rmse, mean_predicted, predictions, target_valid

In [ ]:
# Train models for each region
results = [train_and_evaluate(train_dfs[i], valid_dfs[i]) for i in range(3)]

In [ ]:
# Print results
for i, (rmse, mean_predicted, _, _) in enumerate(results):
    print(f"Region {i}: RMSE = {rmse:.2f}, Avg Predicted Reserves = {mean_predicted:.2f}")

**Model Results (RMSE and predicted reserves):**
- Region 0: RMSE = **37.76**, avg predicted = **92.40** thousand barrels
- Region 1: RMSE = **0.89**, avg predicted = **68.71** thousand barrels
- Region 2: RMSE = **40.15**, avg predicted = **94.77** thousand barrels

**Insights:**
- Region 1 has the highest accuracy (lowest RMSE) but lowest reserves
- Region 2 has the highest average predicted reserves

In [ ]:
# Visualize model performance comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

regions = ['Region 0', 'Region 1', 'Region 2']
rmse_values = [results[i][0] for i in range(3)]
mean_reserves = [results[i][1] for i in range(3)]
colors = ['#3498db', '#e74c3c', '#2ecc71']

# RMSE comparison
bars1 = axes[0].bar(regions, rmse_values, color=colors, edgecolor='white', linewidth=2)
axes[0].set_ylabel('RMSE (thousand barrels)')
axes[0].set_title('Model Prediction Error (RMSE)', fontsize=12, fontweight='bold')
for bar, val in zip(bars1, rmse_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.2f}', ha='center', fontweight='bold')

# Mean predicted reserves
bars2 = axes[1].bar(regions, mean_reserves, color=colors, edgecolor='white', linewidth=2)
axes[1].axhline(111.11, color='red', linestyle='--', linewidth=2, label='Break-even: 111.11')
axes[1].set_ylabel('Average Reserves (thousand barrels)')
axes[1].set_title('Average Predicted Reserves', fontsize=12, fontweight='bold')
axes[1].legend()
for bar, val in zip(bars2, mean_reserves):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/images/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Profit Calculation Setup

In [ ]:
# Profit calculation parameters
BUDGET = 10_000_000_000  # Development budget (rubles)
BARREL_PRICE = 450_000    # Revenue per thousand barrels (rubles)
BEST_WELLS = 200          # Number of best wells to develop
TOTAL_WELLS = 500         # Total wells to explore

# Calculate minimum volume for break-even
min_volume = BUDGET / (BEST_WELLS * BARREL_PRICE)
print(f"Break-even volume: {min_volume:.2f} thousand barrels")

# Compare with average reserves
for i, (_, mean_predicted, _, _) in enumerate(results):
    print(f"Region {i}: Avg reserves = {mean_predicted:.2f}, Break-even = {min_volume:.2f}")

**Break-even Analysis:**
- **Required volume for break-even:** 111.11 thousand barrels
- **Average reserves by region:**
    - Region 0: 92.40 (below threshold)
    - Region 1: 68.71 (significantly below threshold)
    - Region 2: 94.77 (below threshold)

**Insight:** All regions have average reserves below the break-even point. However, we select the TOP 200 wells, not average ones — profit comes from the best deposits.

## Profit and Risk Analysis

In [ ]:
# Profit calculation function
def calculate_profit(target, predictions):
    best_indices = predictions.argsort()[-BEST_WELLS:]
    selected_target = target.iloc[best_indices]
    total_profit = selected_target.sum() * BARREL_PRICE - BUDGET
    return total_profit

# Calculate profit for each region
for i, (_, _, predictions, target) in enumerate(results):
    profit = calculate_profit(target, predictions)
    print(f"Region {i}: Profit = {profit / 1e9:.2f} billion rubles")

**Profit by Region (billion rubles):**
- Region 0: **3.36**
- Region 1: **2.42**
- Region 2: **2.60**

**Insight:** Region 0 shows the highest profit potential. Bootstrap analysis will help assess the reliability of these estimates.

In [ ]:
# Bootstrap profit analysis
def bootstrap_profit(target, predictions, n_simulations=1000):
    np.random.seed(42)
    profits = []
    for _ in range(n_simulations):
        sampled_indices = np.random.randint(0, len(predictions), size=len(predictions))
        sampled_target = target.iloc[sampled_indices]
        sampled_predictions = predictions[sampled_indices]
        profit = calculate_profit(sampled_target, sampled_predictions)
        profits.append(profit)
    return profits

# Analyze risks and profit distribution
for i, (_, _, predictions, target) in enumerate(results):
    profits = bootstrap_profit(target, predictions)
    profits = np.array(profits)
    mean_profit = profits.mean()
    lower_bound = np.percentile(profits, 2.5)
    upper_bound = np.percentile(profits, 97.5)
    risk = (profits < 0).mean() * 100
    print(f"Region {i}: Avg Profit = {mean_profit / 1e9:.2f} billion rubles")
    print(f"95% CI: [{lower_bound / 1e9:.2f}, {upper_bound / 1e9:.2f}] billion rubles")
    print(f"Loss Risk: {risk:.2f}%")

In [ ]:
# Visualize Bootstrap profit distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#3498db', '#e74c3c', '#2ecc71']

bootstrap_results = []
for i, (_, _, predictions, target) in enumerate(results):
    profits = bootstrap_profit(target, predictions)
    profits_b = np.array(profits) / 1e9  # Convert to billions
    bootstrap_results.append(profits_b)

    ax = axes[i]
    ax.hist(profits_b, bins=30, color=colors[i], alpha=0.7, edgecolor='white')
    ax.axvline(profits_b.mean(), color='black', linestyle='-', linewidth=2, label=f'Mean: {profits_b.mean():.2f}B')
    ax.axvline(np.percentile(profits_b, 2.5), color='red', linestyle='--', linewidth=1.5, label='95% CI')
    ax.axvline(np.percentile(profits_b, 97.5), color='red', linestyle='--', linewidth=1.5)
    ax.axvline(0, color='gray', linestyle=':', linewidth=1, label='Break-even')
    ax.set_title(f'Region {i}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Profit (billion rubles)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle('Bootstrap Profit Distribution (1000 Simulations)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('reports/images/bootstrap_profit.png', dpi=150, bbox_inches='tight')
plt.show()

**Bootstrap Analysis Results:**

| Region | Avg Profit (B RUB) | 95% CI | Loss Risk |
|--------|-------------------|--------|-----------|
| **0** | 3.37 | [3.04, 3.74] | 0.00% |
| 1 | 2.42 | [2.42, 2.42] | 0.00% |
| 2 | 2.56 | [2.17, 2.92] | 0.00% |

**Recommendation: Region 0**
- Highest average profit
- Zero loss risk
- Narrow confidence interval indicates stable predictions

## Conclusions

Based on data analysis, model training, profit calculations, and Bootstrap risk assessment, **Region 0 is recommended** for development:

1. **Highest Average Profit:** 3.37 billion rubles (vs 2.42B and 2.56B for other regions)

2. **Stable Predictions:** 95% CI of [3.04, 3.74] billion rubles — narrow interval confirms model reliability

3. **Zero Loss Risk:** All regions show 0% loss probability (threshold was 2.5%)

4. **Trade-off Consideration:** While Region 1 has much lower RMSE (0.89 vs 37.76), Region 0s higher reserve volumes more than compensate for prediction uncertainty

## Project Checklist

Mark completed items with "x":

- [x] Jupyter Notebook opened
- [x] All code runs without errors
- [x] Code cells in execution order
- [x] Step 1: Data prepared
- [x] Step 2: Models trained and validated
- [x] Step 3: Profit calculation prepared
- [x] Step 4: Risks and profit calculated
- [x] Region recommendation provided with justification